# P9-3 · Kiến trúc và hướng mới

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

## Bài tập 5: Lớp MoE tối giản {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class MoE(nn.Module):
    def __init__(self, d=64, d_ff=256, n_experts=8, k=2):
        super().__init__()
        self.k = k
        self.router = nn.Linear(d, n_experts, bias=False)
        self.experts = nn.ModuleList(nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d)) for _ in range(n_experts))

    def forward(self, x):                                  # x: (N, d), N token
        gate = F.softmax(self.router(x), dim=-1)           # (N, E)
        w, idx = gate.topk(self.k, dim=-1)                 # (N, k)
        w = w / w.sum(-1, keepdim=True)
        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            mask = idx == e                                # (N, k)
            rows = mask.any(-1).nonzero(as_tuple=True)[0]
            if len(rows):
                weight = (w * mask).sum(-1)[rows, None]
                out[rows] += weight * expert(x[rows])
        load = torch.bincount(idx.flatten(), minlength=len(self.experts)).float() / idx.numel()
        return out, load

y, load = MoE()(torch.randn(1000, 64))
print(y.shape, load.round(decimals=3))

torch.Size([1000, 64]) tensor([0.1280, 0.1280, 0.1240, 0.1300, 0.1090, 0.1240, 0.1290, 0.1280])
